# Lab 05 - AutoML Nativo (Gemini + Scikit-Learn)
**Disciplina:** Augmented Analytics & AI-Driven Insights  
**Modelo Oficial:** Gemini 3.8 Flash (`gemini-3.8-flash`) via SDK `google-genai`

Este notebook demonstra como criar um agente autônomo de Machine Learning que escreve, executa e valida o próprio pipeline de predição de churn (*White-Box AutoML*) com Scikit-Learn.

In [ ]:
# 1. Instalação da nova biblioteca oficial Google GenAI, leitor de Excel e utilitários
!pip install -q -U google-genai openpyxl tabulate

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google import genai
from google.genai import types
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import re

# 2. Configuração da API Key
# INSTRUÇÃO: Substitua 'COLE_SUA_API_KEY_AQUI' pela sua chave do Google AI Studio
API_KEY = 'COLE_SUA_API_KEY_AQUI'
client = genai.Client(api_key=API_KEY)

In [ ]:
# 3. Carregamento dos dados de Churn
try:
    df = pd.read_excel('Lab 05 - EvasaoClientesClassificados.xlsx')
    print("✅ Base de dados 'Lab 05 - EvasaoClientesClassificados.xlsx' carregada com sucesso!")
    print(f"Total de clientes analisados: {len(df)}")
    display(df.head(3))

    # Preparando o contexto de schema e amostra para o agente de IA
    contexto_colunas = df.dtypes.to_string()
    amostra_dados = df.head(3).to_markdown()
except Exception as e:
    print(f"❌ Erro ao carregar o arquivo Excel. Detalhes: {e}")
    print("Dica: Faça o upload do arquivo 'Lab 05 - EvasaoClientesClassificados.xlsx' no menu lateral do Colab.")

In [ ]:
# 4. FUNÇÃO DE AUTOML NATIVO (O Agente Cientista de Dados)
def executar_automl_nativo(pergunta_negocio="Crie um modelo preditivo de churn"):
    """
    Esta função usa o Gemini para projetar, escrever e executar um pipeline completo
    de Machine Learning White-Box usando Scikit-Learn.
    """
    prompt = f"""
    Você é um Cientista de Dados Sênior especializado em Machine Learning e Scikit-Learn.
    O dataframe 'df' já está carregado e possui as colunas:
    {contexto_colunas}

    Amostra dos dados:
    {amostra_dados}

    A coluna alvo (Target) para prever é 'Perdido?'.

    SUA TAREFA:
    Escreva um código Python completo e auto-contido que:
    1. Remova identificadores únicos ou colunas irrelevantes (como 'ID Cliente').
    2. Trate variáveis categóricas (como 'Sexo', 'Região', 'Nivel Cobertura') usando LabelEncoder ou get_dummies.
    3. Divida os dados em treino e teste (80% treino / 20% teste) com random_state=42.
    4. Treine um modelo de Machine Learning robusto (RandomForestClassifier por padrão, com n_estimators=100).
    5. Imprima a Acurácia e o Relatório de Classificação (classification_report).
    6. Calcule e plote a Matriz de Confusão e o gráfico de Importância das Variáveis (Feature Importance) com matplotlib/seaborn.
    7. RESPONDA OBJETIVAMENTE À PERGUNTA DE NEGÓCIO: {pergunta_negocio}

    REGRAS ESTRITAS:
    - Retorne APENAS o código Python puro dentro de um bloco ```python.
    - Não inclua textos introdutórios ou explicativos fora do bloco de código.
    - Certifique-se de chamar 'plt.show()' para renderizar os gráficos.
    """

    try:
        print("🤖 Agente de AutoML: Desenhando e treinando o pipeline preditivo...\n")
        response = client.models.generate_content(
            model='gemini-3.8-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,
                top_p=0.3
            )
        )

        match = re.search(r'```python\n(.*?)\n```', response.text, re.DOTALL)
        if match:
            codigo = match.group(1).strip()
            print("⚡ Código gerado com sucesso pelo Gemini 3.8 Flash! Executando treinamento...\n")
            exec(codigo, globals())
            print("\n✅ Pipeline de AutoML executado e validado com sucesso!")
        else:
            print("❌ Erro: Não foi possível extrair o bloco de código Python da resposta.")
            print(response.text)
    except Exception as e:
        print(f"⚠️ Erro no processo de AutoML: {e}")
        if 'codigo' in locals():
            print(f"Código tentado:\n{codigo}")

In [ ]:
# 5. EXECUÇÃO DO AUTOML: Descoberta de Drivers de Churn
executar_automl_nativo("Identifique as 3 variáveis que mais influenciam a saída do cliente e avalie a qualidade do modelo.")

In [ ]:
# 6. TESTE DE SENSIBILIDADE: Análise de Nível de Cobertura
executar_automl_nativo("O nível de cobertura influencia diretamente no Churn? Mostre a correlação nos dados.")